# Prefix Classifier Training and Evaluation

This notebook trains the PF00196 prefix classifier from the aligned RR FASTA and the binary label file in `Training_DATA/`.

**WARNING: if the classifier is already trained,it will load it from `checkpoints/Prefix_Classifier/pf00196_prefix_string_classifier.pt` and skip training.**

Set `FORCE_RETRAIN = True` only when you intentionally want to overwrite the saved classifier checkpoint.

In [ ]:
from pathlib import Path
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

SEED = 42
DATA_DIR = Path("Training_DATA")
FASTA_PATH = DATA_DIR / "RR_aligned.fasta"
LABEL_PATH = DATA_DIR / "RR_pf00196_labels.txt"

CHECKPOINT_PATH = Path("checkpoints/Prefix_Classifier/pf00196_prefix_string_classifier.pt")
METRICS_PATH = Path("Prefix_Classifier/pf00196_prefix_string_metrics.csv")
SUMMARY_PATH = Path("Prefix_Classifier/pf00196_prefix_string_summary.json")

FORCE_RETRAIN = False
MAX_EXAMPLES = 100_000  # set to None to use the full RR dataset
BATCH_SIZE = 1024
EPOCHS = 8
LEARNING_RATE = 1e-3
EMBED_DIM = 16
HIDDEN_DIM = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
print(f"Using device: {device}")

## Load The Aligned Dataset

`RR_aligned.fasta` contains the aligned protein strings. `RR_pf00196_labels.txt` contains one binary label per FASTA record.

In [ ]:
def read_fasta(path):
    records = []
    header = None
    chunks = []
    for raw_line in Path(path).read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if header is not None:
                records.append({"header": header, "sequence": "".join(chunks)})
            header = line[1:].strip()
            chunks = []
        else:
            chunks.append(line)
    if header is not None:
        records.append({"header": header, "sequence": "".join(chunks)})
    return pd.DataFrame(records)


def read_binary_labels(path):
    values = []
    for line_number, raw_line in enumerate(Path(path).read_text(encoding="utf-8").splitlines(), start=1):
        line = raw_line.strip()
        if not line:
            continue
        if line not in {"0", "1"}:
            raise ValueError(f"{path}:{line_number} expected 0/1 label, found {line!r}")
        values.append(int(line))
    return np.asarray(values, dtype=np.int64)


df = read_fasta(FASTA_PATH)
labels = read_binary_labels(LABEL_PATH)

if len(df) != len(labels):
    raise ValueError(f"FASTA records ({len(df):,}) and labels ({len(labels):,}) do not match.")

df["target"] = labels
df["seq_len"] = df["sequence"].str.len()
if df["seq_len"].nunique() != 1:
    raise ValueError("The classifier expects a fixed-length alignment.")

sequence_length = int(df["seq_len"].iloc[0])
if MAX_EXAMPLES is not None and len(df) > MAX_EXAMPLES:
    df = (
        df.groupby("target", group_keys=False)
        .apply(lambda group: group.sample(
            n=min(len(group), max(1, int(round(MAX_EXAMPLES * len(group) / len(labels))))),
            random_state=SEED,
        ))
        .sample(frac=1.0, random_state=SEED)
        .head(MAX_EXAMPLES)
        .reset_index(drop=True)
    )

print(f"Rows used: {len(df):,}")
print(f"Aligned length: {sequence_length}")
print(f"Positive rate: {df['target'].mean():.4%}")
display(df.head())

## Tokenizer And Splits

The tokenizer is character-level and includes a pad token used to represent missing prefix positions.

In [ ]:
PAD_SYMBOL = "?"
chars = sorted(set("".join(df["sequence"].tolist())))
if PAD_SYMBOL in chars:
    raise ValueError(f"Pad symbol {PAD_SYMBOL!r} is already present in the data.")
chars = chars + [PAD_SYMBOL]
stoi = {ch: idx for idx, ch in enumerate(chars)}
itos = {idx: ch for ch, idx in stoi.items()}
pad_token = stoi[PAD_SYMBOL]
vocab_size = len(stoi)


def encode(sequence):
    return [stoi[ch] for ch in sequence]


def decode(token_ids):
    return "".join(itos[int(idx)] for idx in token_ids)


def stratified_split_indices(targets, test_size=0.2, val_size=0.2, seed=SEED):
    rng = np.random.default_rng(seed)
    targets = np.asarray(targets)
    train_parts, val_parts, test_parts = [], [], []
    for label in np.unique(targets):
        idx = np.flatnonzero(targets == label)
        rng.shuffle(idx)
        n_test = int(round(len(idx) * test_size))
        n_val = int(round((len(idx) - n_test) * val_size))
        test_parts.append(idx[:n_test])
        val_parts.append(idx[n_test:n_test + n_val])
        train_parts.append(idx[n_test + n_val:])
    train_idx = np.concatenate(train_parts)
    val_idx = np.concatenate(val_parts)
    test_idx = np.concatenate(test_parts)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)
    return train_idx, val_idx, test_idx


sequences = df["sequence"].tolist()
targets = df["target"].to_numpy(np.int64)
train_idx, val_idx, test_idx = stratified_split_indices(targets)

train_sequences = [sequences[i] for i in train_idx]
val_sequences = [sequences[i] for i in val_idx]
test_sequences = [sequences[i] for i in test_idx]
train_targets = targets[train_idx]
val_targets = targets[val_idx]
test_targets = targets[test_idx]

print(f"Tokenizer vocabulary: {chars}")
print(f"vocab_size={vocab_size}, pad_token={pad_token}")
print(f"train/val/test: {len(train_idx):,}/{len(val_idx):,}/{len(test_idx):,}")

## Model And Metrics

The model sees a prefix padded to the full aligned length. During training, each example is shown at a random prefix length so that the classifier learns to score partial sequences.

In [ ]:
class PrefixStringClassifier(nn.Module):
    def __init__(self, vocab_size, block_size, pad_token, embed_dim=16, hidden_dim=16):
        super().__init__()
        self.block_size = block_size
        self.pad_token = pad_token
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_token)
        self.classifier = nn.Sequential(
            nn.Linear(block_size * embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, tokens, prefix_lengths=None):
        embedded = self.embedding(tokens)
        return self.classifier(embedded.reshape(tokens.shape[0], -1)).squeeze(1)


def encode_prefix_batch(batch_sequences, contexts, block_size):
    tokens = torch.full((len(batch_sequences), block_size), pad_token, dtype=torch.long)
    lengths = torch.as_tensor(contexts, dtype=torch.long)
    for row, (sequence, context) in enumerate(zip(batch_sequences, contexts)):
        context = int(context)
        if context > 0:
            tokens[row, :context] = torch.tensor(encode(sequence[:context]), dtype=torch.long)
    return tokens, lengths


def metric_dict(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {
        "accuracy": (tp + tn) / max(len(y_true), 1),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


@torch.inference_mode()
def collect_logits(model, sequences, contexts, batch_size=BATCH_SIZE):
    model.eval()
    out = {}
    for context in contexts:
        logits = []
        context_array = np.full(batch_size, int(context), dtype=np.int64)
        for start in range(0, len(sequences), batch_size):
            batch_sequences = sequences[start:start + batch_size]
            batch_contexts = context_array[:len(batch_sequences)]
            x, lengths = encode_prefix_batch(batch_sequences, batch_contexts, sequence_length)
            logits.append(model(x.to(device), lengths.to(device)).cpu().numpy())
        out[int(context)] = np.concatenate(logits)
    return out


def best_accuracy_threshold(logits, targets):
    logits = np.asarray(logits, dtype=np.float32)
    targets = np.asarray(targets, dtype=np.int64)
    n = len(logits)
    if n == 0:
        return 0.0

    order = np.argsort(logits)[::-1]
    sorted_logits = logits[order]
    sorted_targets = targets[order]
    correct = int((targets == 0).sum())
    best_correct = correct
    best_threshold = float(np.nextafter(logits.max(), np.inf))
    idx = 0

    while idx < n:
        current_logit = sorted_logits[idx]
        while idx < n and sorted_logits[idx] == current_logit:
            correct += 1 if sorted_targets[idx] == 1 else -1
            idx += 1
        if correct > best_correct:
            best_correct = correct
            best_threshold = float(current_logit)
    return best_threshold


def fit_thresholds(model, sequences, targets, contexts):
    logits_by_context = collect_logits(model, sequences, contexts)
    thresholds = {0: None}
    for context in contexts:
        if context == 0:
            continue
        thresholds[int(context)] = best_accuracy_threshold(logits_by_context[int(context)], targets)
    return thresholds


def evaluate_by_prefix_length(model, sequences, targets, contexts, thresholds):
    rows = []
    targets = np.asarray(targets, dtype=np.int64)
    logits_by_context = collect_logits(model, sequences, [c for c in contexts if c > 0])
    majority_prediction = int(np.mean(targets) >= 0.5)
    for context in contexts:
        if context == 0:
            pred = np.full(len(targets), majority_prediction, dtype=np.int64)
            threshold = np.nan
        else:
            threshold = float(thresholds.get(int(context), 0.0))
            pred = (logits_by_context[int(context)] >= threshold).astype(np.int64)
        row = {"prefix_length": int(context), "threshold": threshold, "n_eval": int(len(targets))}
        row.update(metric_dict(targets, pred))
        rows.append(row)
    return pd.DataFrame(rows)

## Train Or Load The Classifier

The warning above is implemented here: by default, an existing checkpoint is loaded and training is skipped.

In [ ]:
def train_classifier():
    model = PrefixStringClassifier(vocab_size, sequence_length, pad_token, EMBED_DIM, HIDDEN_DIM).to(device)
    pos = max(float(train_targets.sum()), 1.0)
    neg = max(float(len(train_targets) - train_targets.sum()), 1.0)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg / pos], device=device))
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    best_state = None
    best_val_f1 = -1.0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        start_time = time.time()
        order = rng.permutation(len(train_sequences))
        losses = []
        for start in range(0, len(order), BATCH_SIZE):
            batch_idx = order[start:start + BATCH_SIZE]
            batch_sequences = [train_sequences[i] for i in batch_idx]
            batch_targets = torch.tensor(train_targets[batch_idx], dtype=torch.float32, device=device)
            batch_contexts = rng.integers(1, sequence_length + 1, size=len(batch_sequences))
            x, lengths = encode_prefix_batch(batch_sequences, batch_contexts, sequence_length)
            logits = model(x.to(device), lengths.to(device))
            loss = criterion(logits, batch_targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item()))

        quick_contexts = sorted(set([1, 5, 10, 20, 40, 60, 80, sequence_length]))
        quick_thresholds = fit_thresholds(model, val_sequences, val_targets, quick_contexts)
        quick_metrics = evaluate_by_prefix_length(model, val_sequences, val_targets, quick_contexts, quick_thresholds)
        val_f1 = float(quick_metrics["f1"].mean())
        print(
            f"epoch={epoch:02d} loss={np.mean(losses):.4f} "
            f"mean_val_f1={val_f1:.4f} seconds={time.time() - start_time:.1f}"
        )
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model


if CHECKPOINT_PATH.exists() and not FORCE_RETRAIN:
    print(f"WARNING: classifier already trained. Loading from {CHECKPOINT_PATH} and skipping training.")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    stoi = checkpoint["stoi"]
    itos = checkpoint["itos"]
    vocab_size = checkpoint["vocab_size"]
    pad_token = checkpoint["pad_token"]
    PAD_SYMBOL = checkpoint.get("pad_symbol", PAD_SYMBOL)
    sequence_length = checkpoint["sequence_length"]
    EMBED_DIM = checkpoint.get("embed_dim", EMBED_DIM)
    HIDDEN_DIM = checkpoint.get("hidden_dim", HIDDEN_DIM)
    model = PrefixStringClassifier(vocab_size, sequence_length, pad_token, EMBED_DIM, HIDDEN_DIM).to(device)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    thresholds = checkpoint.get("thresholds_by_context")
else:
    model = train_classifier()
    thresholds = None

## Metrics As A Function Of Prefix Length

Thresholds are fit on the validation split, then the final table and plot are computed on the held-out test split.

In [ ]:
contexts = list(range(0, sequence_length + 1))
if thresholds is None or any((context > 0 and context not in thresholds) for context in contexts):
    thresholds = fit_thresholds(model, val_sequences, val_targets, contexts)

metrics_df = evaluate_by_prefix_length(model, test_sequences, test_targets, contexts, thresholds)
metrics_df.to_csv(METRICS_PATH, index=False)

display(metrics_df.head())
display(metrics_df.tail())
print(f"Saved metrics to {METRICS_PATH}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(metrics_df["prefix_length"], metrics_df["accuracy"], label="Accuracy", linewidth=2)
ax.plot(metrics_df["prefix_length"], metrics_df["precision"], label="Precision", linewidth=1.8)
ax.plot(metrics_df["prefix_length"], metrics_df["recall"], label="Recall", linewidth=1.8)
ax.plot(metrics_df["prefix_length"], metrics_df["f1"], label="F1", linewidth=2)
ax.axhline(test_targets.mean(), color="black", linestyle="--", linewidth=1, label="Positive rate")
ax.set_title("Classifier Metrics by Prefix Length")
ax.set_xlabel("Prefix length")
ax.set_ylabel("Metric")
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Save Checkpoint

The classifier checkpoint stores the model weights, tokenizer, and validation thresholds.

In [ ]:
checkpoint = {
    "state_dict": {key: value.detach().cpu() for key, value in model.state_dict().items()},
    "stoi": stoi,
    "itos": itos,
    "vocab_size": vocab_size,
    "pad_token": pad_token,
    "pad_symbol": PAD_SYMBOL,
    "sequence_length": sequence_length,
    "target_label": "PF00196",
    "embed_dim": EMBED_DIM,
    "hidden_dim": HIDDEN_DIM,
    "thresholds_by_context": thresholds,
}
torch.save(checkpoint, CHECKPOINT_PATH)

summary = {
    "checkpoint": str(CHECKPOINT_PATH),
    "metrics_csv": str(METRICS_PATH),
    "n_examples": int(len(df)),
    "sequence_length": int(sequence_length),
    "positive_rate": float(df["target"].mean()),
    "best_test_accuracy": float(metrics_df["accuracy"].max()),
    "best_test_f1": float(metrics_df["f1"].max()),
    "full_length": metrics_df.iloc[-1].to_dict(),
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"Saved classifier to {CHECKPOINT_PATH}")
print(f"Saved summary to {SUMMARY_PATH}")